In [32]:
# 목표: 반려동물이 입양되는지 예측
# 내 목표는 코드 작동원리를 이해하는가

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers

In [33]:
dataframe = pd.read_csv('datasets/petfinder_mini_extracted/petfinder-mini/petfinder-mini.csv')
dataframe.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,Description,PhotoAmt,AdoptionSpeed
0,Cat,3,Tabby,Male,Black,White,Small,Short,No,No,Healthy,100,Nibble is a 3+ month old ball of cuteness. He ...,1,2
1,Cat,1,Domestic Medium Hair,Male,Black,Brown,Medium,Medium,Not Sure,Not Sure,Healthy,0,I just found it alone yesterday near my apartm...,2,0
2,Dog,1,Mixed Breed,Male,Brown,White,Medium,Medium,Yes,No,Healthy,0,Their pregnant mother was dumped by her irresp...,7,3
3,Dog,4,Mixed Breed,Female,Black,Brown,Medium,Short,Yes,No,Healthy,150,"Good guard dog, very alert, active, obedience ...",8,2
4,Dog,1,Mixed Breed,Male,Black,No Color,Medium,Short,No,No,Healthy,0,This handsome yet cute boy is up for adoption....,3,2


In [34]:
# 표 정리

dataframe['target'] = np.where(dataframe["AdoptionSpeed"]==4, 0, 1)
# np.where(조건, 참일 때 값, 거짓일 때 값)
dataframe = dataframe.drop(columns=['AdoptionSpeed', 'Description'])

dataframe.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,PhotoAmt,target
0,Cat,3,Tabby,Male,Black,White,Small,Short,No,No,Healthy,100,1,1
1,Cat,1,Domestic Medium Hair,Male,Black,Brown,Medium,Medium,Not Sure,Not Sure,Healthy,0,2,1
2,Dog,1,Mixed Breed,Male,Brown,White,Medium,Medium,Yes,No,Healthy,0,7,1
3,Dog,4,Mixed Breed,Female,Black,Brown,Medium,Short,Yes,No,Healthy,150,8,1
4,Dog,1,Mixed Breed,Male,Black,No Color,Medium,Short,No,No,Healthy,0,3,1


In [41]:
train, val, test = np.split(dataframe.sample(frac=1), [int(0.8*(len(dataframe))), int(0.9*len(dataframe))])
# dataframe.sample(frac=1) 나누기전에 섞기.
# (샘플링은 뽑다/ 추출하다 인데 특징이 무작위로 뽑는거라서 셔플이랑 다를 게 없다.(pandas에선 이걸쓰삼. tf에서는 셔플))
# (frac은 비율의 약자. 0.5는 50%, 1은 100%)
# split은 칼자국.

# train = train.reset_index(drop=True) # 인덱스 재정렬
# drop=True는 옛날 번호표들 버려도된다는뜻.

print(len(train))
print(len(val))
print(len(test))

9229
1154
1154


/Users/jeong-eunbyeol/Desktop/깃허부/.venv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [42]:
# class로도.. 재조립해보기.

# 데이터셋으로 가공 ! !
def df_to_dataset(dataframe, shuffle=True, batch_size=32):
    df = dataframe.copy()
    labels = df.pop('target')
    df = {key : value.to_numpy()[:, tf.newaxis] for key, value in dataframe.items()}
    
    ds = tf.data.Dataset.from_tensor_slices((dict(df), labels))
    # tf.data.Dataset.from_tensor_slices(문제, 정답) 이렇게 하면 
    # 결과물이 ((나이: 성별: 어쩌구:), 정답:) 이롷게 묶어서 줌. (이게 데이터셋)
    
    # 텐서란. 데이터를 답는 다차원 배열
    # 데이터셋은 사람1데이터들, 사람2데이터들... 
    
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe))
    ds = ds.batch(batch_size)
    ds = ds.prefetch(batch_size)
    return ds

In [55]:
# 확인용 예시
batch_size = 5
train_ds = df_to_dataset(train, batch_size=batch_size)

[(train_features, label_batch)] = train_ds.take(1)
print(list(train_features.keys()))
print(train_features['Age'])
print(label_batch)

['Type', 'Age', 'Breed1', 'Gender', 'Color1', 'Color2', 'MaturitySize', 'FurLength', 'Vaccinated', 'Sterilized', 'Health', 'Fee', 'PhotoAmt', 'target']
tf.Tensor(
[[1]
 [2]
 [6]
 [4]
 [5]], shape=(5, 1), dtype=int64)
tf.Tensor([0 1 1 0 1], shape=(5,), dtype=int64)


이 코드 안에서 아래 네 개의 전처리 레이어를 사용한다.

| Keras 전처리 레이어 | 설명 |
|:---|:---|
| tf.keras.layers.Normalization | 정규화수행. 데이터 분포를 표준화한다. |
| tf.keras.layers.CategoryEncoding | 정수를 원-핫, 멀티-핫 또는 TF-IDF으로 바꾼다. |
| tf.keras.layers.StringLookup | 문자열 범주형 값 -> 정수 인덱스 |
| tf.keras.layers.IntegerLookup | 정수 범주형 값 -> 정수 인덱스 |
